# 🧠 License Plate Detection — YOLOv8s Training on Colab

Fine-tune YOLOv8s on the **Indian Number Plate Annotation** dataset from Kaggle
(15,338 images, native YOLO format, single class `LicensePlate`).

Dataset: https://www.kaggle.com/datasets/sunrajbishnolia/license-plate-detection

**Run top-to-bottom.** Estimated time: ~1.5–3 hours on a free T4 GPU (early stopping usually ends it sooner).

## ⚙️ Before you start

1. **Runtime → Change runtime type → T4 GPU** (this notebook will not train on CPU).
2. **No Kaggle API token needed** — the dataset is public and downloads via `kagglehub`.
3. Mount Google Drive when asked — the final `best.pt` is copied there.
4. Checkpoints are saved in the Colab VM every 10 epochs; the final export goes to Drive.
   Free Colab sessions can disconnect, so keep the tab open while training runs.

### 1 · Mount Google Drive (~10 seconds)

Used for the final `best.pt` export.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2 · Install dependencies (~30 seconds)

In [ ]:
!pip install -q ultralytics kagglehub
import ultralytics
print("ultralytics", ultralytics.__version__)

### 3 · Verify the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
print(f"\n✅ GPU ready: {torch.cuda.get_device_name(0)}")

### 4 · Download the dataset (~2–5 minutes, ~870 MB)

Public dataset via `kagglehub` — no token, no upload. The zip nests everything
inside a `License Plate Detection/` folder; we locate that automatically.

In [ ]:
import kagglehub
from pathlib import Path

# Downloads + extracts into the kagglehub cache (public dataset — no auth)
cache_root = Path(kagglehub.dataset_download("sunrajbishnolia/license-plate-detection"))

# Descend into the nested folder that actually holds train/valid/test
DATA_ROOT = cache_root
for cand in DATA_ROOT.rglob("train"):
    if cand.is_dir() and (cand.parent / "valid").exists():
        DATA_ROOT = cand.parent
        break

print("Dataset root:", DATA_ROOT)
print("Contents:", [p.name for p in DATA_ROOT.iterdir()])

### 5 · Fix `data.yaml` if needed

The zip's shipped `data.yaml` contains literal `.../train/images` placeholder paths
(verified on dataset v2) — we overwrite it with real paths.

In [ ]:
data_yaml = DATA_ROOT / "data.yaml"

existing = data_yaml.read_text() if data_yaml.exists() else ""
if "..." in existing or not existing:
    data_yaml.write_text(
        f"path: {DATA_ROOT}\n"
        "train: train/images\n"
        "val: valid/images\n"
        "test: test/images\n"
        "\n"
        "nc: 1\n"
        "names: ['LicensePlate']\n"
    )
    print("📝 Wrote valid data.yaml")

print(data_yaml.read_text())

## 📊 About the dataset

- **15,338 annotated images**: 10,736 train / 3,068 val / 1,534 test
- **Single class**: `LicensePlate`
- **Native YOLO format** — one `.txt` per image with normalized `class cx cy w h` rows
- Indian plates across multiple states, day/night, shadows, glare, plus
  augmented samples (blur, rotation, brightness)

### 6 · Verify the dataset

In [ ]:
for split in ("train", "valid", "test"):
    imgs = list((DATA_ROOT / split / "images").glob("*"))
    lbls = list((DATA_ROOT / split / "labels").glob("*.txt"))
    print(f"{split:6s}: {len(imgs):6,d} images · {len(lbls):6,d} labels")
print("\nData config:\n", data_yaml.read_text())

### 7 · Peek at annotated samples (~20 seconds)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

img_dir = DATA_ROOT / "train" / "images"
lbl_dir = DATA_ROOT / "train" / "labels"
samples = random.sample(sorted(img_dir.glob("*")), 9)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = lbl_dir / (img_path.stem + ".txt")
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
            x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (46, 204, 113), 2)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
fig.suptitle("Random training images with ground-truth bounding boxes", fontsize=14)
plt.tight_layout()
plt.show()

## 🏋️ 8 · Train YOLOv8s (main cell — ~1.5–3 hours)

`yolov8s` (small) on a T4 16GB: `batch=16` fits comfortably with mixed precision.
Checkpoints land in the Colab VM every 10 epochs so a disconnect costs you at most
10 epochs. Early stopping (`patience=10`) halts training if validation mAP plateaus.

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLOv8s (small) — Colab T4 handles this easily
model = YOLO("yolov8s.pt")

# Train on the Kaggle Indian Number Plate dataset
results = model.train(
    data=str(data_yaml),                 # Path to data.yaml (fixed in step 5)
    epochs=80,
    batch=16,          # T4 16GB can handle this
    imgsz=640,
    amp=True,          # Mixed precision
    patience=10,       # Early stopping
    device=0,
    workers=2,
    project="runs/detect",
    name="license_plate",
    exist_ok=True,
    verbose=True,

    # Checkpoint every 10 epochs (stored in the Colab VM — best.pt goes to Drive at the end)
    save_period=10,
)

print("✅ Training complete!")
print("Best weights saved at: runs/detect/license_plate/weights/best.pt")

### 9 · Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

run_dir = Path("runs/detect/license_plate")
res = pd.read_csv(run_dir / "results.csv")
res.columns = [c.strip() for c in res.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(res["epoch"], res["train/box_loss"], label="train")
axes[0, 0].plot(res["epoch"], res["val/box_loss"], label="val")
axes[0, 0].set_title("Box loss")
axes[0, 0].legend()

axes[0, 1].plot(res["epoch"], res["train/cls_loss"], label="train")
axes[0, 1].plot(res["epoch"], res["val/cls_loss"], label="val")
axes[0, 1].set_title("Classification loss")
axes[0, 1].legend()

axes[1, 0].plot(res["epoch"], res["metrics/mAP50(B)"], color="#2ecc71")
axes[1, 0].set_title("mAP@50")

axes[1, 1].plot(res["epoch"], res["metrics/mAP50-95(B)"], color="#4a90d9")
axes[1, 1].set_title("mAP@50-95")

for ax in axes.flat:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
fig.suptitle("YOLOv8s training curves — license plate detection", fontsize=14)
plt.tight_layout()
plt.show()

last = res.iloc[-1]
print(f"Final epoch {int(last['epoch'])}: "
      f"mAP@50 = {last['metrics/mAP50(B)']:.3f} · "
      f"mAP@50-95 = {last['metrics/mAP50-95(B)']:.3f}")

### 10 · Formal validation on the held-out set (~1 minute)

In [ ]:
best = YOLO("runs/detect/license_plate/weights/best.pt")
metrics = best.val(data=str(data_yaml), split="test")

print("\n📊 Test-set results")
print(f"   mAP@50    : {metrics.box.map50:.3f}")
print(f"   mAP@50-95 : {metrics.box.map:.3f}")
print(f"   Precision : {metrics.box.mp:.3f}")
print(f"   Recall    : {metrics.box.mr:.3f}")

### 11 · Sample predictions (~30 seconds)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

test_imgs = random.sample(sorted((DATA_ROOT / "test" / "images").glob("*")), 6)
preds = best.predict([str(p) for p in test_imgs], conf=0.5, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, r in zip(axes.flat, preds):
    img = cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2RGB)
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(img, (x1, y1), (x2, y2), (46, 204, 113), 2)
        cv2.putText(img, f"{float(box.conf[0]):.2f}", (x1, max(0, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (46, 204, 113), 2)
    ax.imshow(img)
    ax.axis("off")
fig.suptitle("Trained model predictions on test images (conf > 0.5)", fontsize=14)
plt.tight_layout()
plt.show()

### 12 · Export `best.pt` to Google Drive

In [ ]:
!cp runs/detect/license_plate/weights/best.pt /content/drive/MyDrive/best.pt
!ls -lh /content/drive/MyDrive/best.pt
print("✅ Copied to Google Drive")

## 🏁 Next steps

1. Download `best.pt` from **Google Drive → MyDrive** to your PC.
2. Back up the current Level 0 weights (for before/after comparison):
   `copy models\best.pt models\best_level0.pt`
3. Replace: put the new weights at `models/best.pt`.
4. Re-run the image test and benchmarks locally:

```bash
python scripts/test_images.py --num 150 --seed 7
python scripts/benchmark.py
```

Expected Level 1 detection quality: **~90% mAP@50** on Indian plates (vs ~75% Level 0).